Comenzamos cargando los archivos json con los diccionarios que contienen la información de reddit

In [ ]:
import json
from google.colab import drive
drive.mount('/content/drive')


with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_wallstreet_corregido", "r", encoding="utf-8") as f:
    result_walstreet_corregido = json.load(f)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_soccer_corregido", "r", encoding="utf-8") as f:
    result_soccer_corregido = json.load(f)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_nba_corregido", "r", encoding="utf-8") as f:
    result_nba_corregido = json.load(f)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_unpopular_corregido", "r", encoding="utf-8") as f:
    result_unpopular_corregido = json.load(f)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_fortnite_corregido", "r", encoding="utf-8") as f:
    result_fortnite_corregido = json.load(f)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_elden_corregido", "r", encoding="utf-8") as f:
    result_elden_corregido = json.load(f)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Creamos el prompt que leera el modelo. Como es de un estilo ZSL no deberemos añadirle ejemplos de ejecución para que aprenda, unicamente le diremos que cree resumenes sobre el submissión seleccionado

In [ ]:
zero_shot_prompt = """
Eres un modelo cuya tarea es resumir el texto recibido en una sola oración.

Por favor, resume el siguiente texto:"""

In [ ]:
!pip install transformers huggingface_hub
!pip install -U bitsandbytes

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Obtener el token de secretos de Colab
hf_token = userdata.get('HGKEY-1')

# Iniciar sesión con el token
login(token=hf_token)

## Gemma-3-1b-it

Para la primera creación de resumenes hemos usado el modelo SLM gemma-3-1b-it. Dentro de la pagina de hugging fece hemos comprobado la información relevante a este modelo para evaluar tanto los puntos positivos a la hora de usarlo como los negativos.

- como vemos usa it que nos indica que esta entrenado para usar enfoque ZSL recibiendo ordenes directas.
- Gemma 3 en general procesa mucho mejor la concatenación de textos largos debido a su arquitectura.

- Como vemos en la practica responde como un chatbot añadiendo texto al principio que deberemos eliminar.
- este modelo puede llegar a ser muy extractivo lo que le haga coger texto literal en vez de recoger inforamción relevante.

In [ ]:
# Importamos las librerias necesarias
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM

# Definimos el checkpoint que vamos a usar
model_path = "google/gemma-3-1b-it"

# Configuración la cuantización de 4-bit.
# La cuantización de 4 bits ayuda a optimizar el uso de memoria de la GPU,
# haciendo que el modelo sea más eficiente.
quantization_config = BitsAndBytesConfig(load_in_4bit = True)

# Cargamos el tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Cargamos el modelo.
# Lo cuantizamos a 4 bits para optimizar el uso de memoria.
# Lo ponemos en modo de evaluación (eval) para que se comporte
# en modo inferencia (no dropout, uso de estadísticas acumuladas).
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config = quantization_config,
    device_map = "auto"
).eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Como nos encontramos en algunos casos con enormes cantidades de comentarios que dificultan las acciones del modelo nos hemos encontrado en la necesidad de paralelizar este proceso pensando en su ejecución con cpu usando el conocimiento y codigo obtenido en otras practicas como procesamiento paralelo de datos. Esto principalmente se deverá plantear en la función generate_response, la cual creará el resumen del texto añadido.

In [ ]:
def generate_response(base_prompt,
                      texts,
                      model=model,
                      tokenizer=tokenizer,
                      temp=0.9,
                      t_p=0.95,
                      t_k=35,
                      max_tokens=1000):

    # 1. Configuración CRÍTICA para generar por lotes (batching) si el batch no está completo se añade el termino <pad> para complentarlo (no crea cambios en el modelo)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Si por error se pasa un texto suelto (string), lo metemos en una lista
    if isinstance(texts, str):
        texts = [texts]

    # 2. Aplicar el template a todos los textos (sin tokenizar todavía)
    # utliza un estilo de mensajería por el cual un usuario manda cotenido a una maquina que hace el maximo trabajo posible
    prompts = []
    for text in texts:
        messages = [{"role": "user", "content": f"{base_prompt} {text}"}]
        prompt_formateado = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False # Lo mantenemos como texto de momento
        )
        prompts.append(prompt_formateado)

    # 3. Tokenizar todos los textos de golpe con padding y truncation (evita errores de longitud)
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048 # Límite seguro para no saturar la RAM de la GPU
    ).to(model.device)

    # 4. Generar todas las respuestas simultáneamente en la GPU
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=temp,
        top_k=t_k,
        top_p=t_p,
        do_sample=True
    )

    # 5. Extraer SOLO los tokens nuevos generados (mucho más rápido que el '.split()')
    input_length = inputs.input_ids.shape[1]
    tokens_generados = outputs[:, input_length:]

    # 6. Decodificar todo el lote de golpe
    responses = tokenizer.batch_decode(tokens_generados, skip_special_tokens=True)

    return [resp.strip() for resp in responses]

Aqui creamos la función que limpia el resumen para poder eliminar los inicios de texto como "el texto describe".

In [ ]:
import re

def limpiar_resumen(texto, min_len=40, max_len = 600):
    # 1. Eliminar introducciones del modelo (ignorando mayúsculas/minúsculas)
    patrones_inicio = r"^(the text describes|el texto describe|el texto parecer ser|el narrador describe|the author describes|The author considers|The author is reflecting on|The author is describing|The text summarizes)\s*:?\s*"
    texto = re.sub(patrones_inicio, "", texto, flags=re.IGNORECASE)

    # 2. Reemplazar saltos de línea y espacios múltiples por un solo espacio
    texto = re.sub(r'\s+', ' ', texto)

    #Evaluamos si el resumen es aceptable o no comprobando que el tamaño del texto originado está dentro de unos margenes lógicos
    if min_len <= len(texto) <= max_len:
        clasificacion = "aceptable"
    else:
        clasificacion = "no aceptable"

    # 3. Eliminar espacios sobrantes al principio y al final
    return texto.strip(), clasificacion

Ahora comenzamos el proceso de resumir el texto. Para eso primero deberemos recoger la información del submissión como el titulo y la descripción que aparecen en el diccionario. A continuación recogemos todos los comentarios que aparecen en este submission y creamos el texto completo con el titulo, la descripción y los comentarios que formarán el texto.

In [ ]:
import pandas as pd
subreddit_list = [result_elden_corregido, result_fortnite_corregido, result_walstreet_corregido, result_unpopular_corregido, result_nba_corregido, result_soccer_corregido]
# 1. Recopilar todas las submissions y preparar sus textos
todas_las_submissions = []
textos_a_procesar = []

for subreddit in subreddit_list:
    for submission in subreddit.get("submissions", []):
        title = submission.get("title_preprocessed") or ""
        description = submission.get("description") or ""

        comentarios_validos = [
            c.get("body_preprocessed") for c in submission.get("comments", [])
            if c.get("body_preprocessed") and c.get("body_preprocessed") != "[removed]" and c.get("author") != "AutoModerator"
        ]

        comments_text = ". ".join(comentarios_validos) + ". " if comentarios_validos else ""
        texto_completo = f"{title}. {description}. {comments_text}".strip()

        # Guardamos la referencia al diccionario original y el texto
        todas_las_submissions.append(submission)
        textos_a_procesar.append(texto_completo)

# 2. Procesar en lotes (batching) usando tu generate_response optimizado
batch_size = 8
lista_res_gemma = []
#Hacemos bucle de unicamente los diez primeros hilos para poder compararlos, no añadimos mas por que tarda demasiado
for i in range(0, 10, batch_size):
    # Extraemos el bloque actual de textos y sus submissions
    lote_textos = textos_a_procesar[i : i + batch_size]
    lote_submissions = todas_las_submissions[i : i + batch_size]

    # Generamos los resúmenes de golpe en la GPU
    lote_resumenes = generate_response(zero_shot_prompt, lote_textos, max_tokens=100)

    # 3. Guardamos cada resumen directamente en su diccionario original
    for submission, resumen in zip(lote_submissions, lote_resumenes):

      resumen_limpiado, clasificacion = limpiar_resumen(resumen)
      submission['resumen_gemma'] = resumen_limpiado

      #guardamos los 10 primero resumenes de los hilos en una lista para evaluarlos a posteriori
      if len(lista_res_gemma) < 10:
        lista_res_gemma.append((resumen_limpiado, clasificacion))

Ahora imprimimos tanto el texto original como el resumen creado por gemma para poder hacer una evuluación de estos.

In [ ]:
for i in range(10):
  print(f"Texto {i + 1}: {textos_a_procesar[i]}")

Texto 1: did your opinion on the dec change since launch would you rate it different now than in the first months. . at first i thought it was peak then i realized its even more peak then peak. it's good boss fights can be as fun as those of the base game however the lack of cut scenes make it seem like a bit of a rushed job nonetheless overall it was well done. i like the base game more. i played it back to back recently and the one thing i noticed on the and run is that the world did feel pretty empty relative to the base game everything else good or bad felt the same. same as launch many things added too few explained but it managed to make me emphasize with a character when the main game couldn't marinas story hits deep. no of of back then of of now. it is what a dec should be. still love it. over time it gets kinda tiring playing it the first time with a new character and having to collect the fragments but i like how fast you rush through it on no cycles expect for mira. i liked 

In [ ]:
for i in range(len(lista_res_gemma)):
  resumen = lista_res_gemma[i][0]
  clasificacion = lista_res_gemma[i][1]
  print(f"Resumen {i+1} ({clasificacion}): {resumen}")

Resumen 1 (aceptable): a deep appreciation for the "dec" (the core gameplay of the dec game) and its unique aesthetic. The author feels a strong connection to the "base game" due to its simplicity, and finds the dec itself to be a highly valued and interesting addition to the game, considering it an "expansion" to the base experience. They highlight the enjoyment of collecting fragments, particularly the beautiful "shadow fragments," and that the focus on dungeons, especially the "gems" (golden
Resumen 2 (aceptable): El resumen es: El autor experimentó con una estrategia de "cacharra" (dejar la pelea en el momento justo) contra jefes o enemigos, lo que le permitía obtener la mejor experiencia y poder, pero se veía superado por su propia falta de velocidad, y a veces, la falta de la reacción de la IA. Esto se veía reflejado en la pérdida de la información en la pantalla, y el uso de la "best" o "prime" of la defensa
Resumen 3 (aceptable): el estado de ánimo de un "fan" de la YouTuber Ka

Para empezar, el hecho de tener que cortar parte del texto por que no tiene nada que ver con el resumen en si, sino que es un encabezado, hace que estemos más limitado, perdemos tokens que podrían usarse en información importante unicamente en decirnos que es un resumen, algo que ya sabemos ya que somo nostros los que le pedimos la tarea.
Además podemos ver que al final de los resumenes las frases no están terminadas del todo, sea bien por el límite de tokens o por la manera de resumir del modelo no es asumible dejar información a medias por lo que esto es un punto negativo más que vemos para gemma.
Como punto a favor podemos dar que todos los resumenes han sido reconocidos como aceptables, por lo que entran dentro de un margen logico de resumen.

### Evaluacion
Para hacer una evaluación del resumen usaremos el octavo submission ya que es el más corto, los demás son demasiado largos y unicamente queremos comprobar si ha capatado la información relevante.
Si leemos el texto vemos que es un hilo que habla sobre la creación de un personaje de videojuegos el cual tiene un aspecto entre voldemort y el joker al cual. El usuario pide que le propongan nombres mientras el dice dos como vole her o bala rent que podrían encajar. Ya con esto vemos que el resumen en ningún momento habla que el usuario está solicitando que se le aporte un nombre al personaje, unicamente habla de las comparaciones que se le dan y siquiera que de los nombres que aportan los usuarios. Por lo tanto este resumen no podría considerarse adecuado.

Debido a que sería muy tedioso hacer una comparación para cada resumen vamos a usar los modelos anteriores para crear un juez que decida si el resumen es correcto o no y evalue el proceso.

In [ ]:
import re
import torch

def evaluar_resumen(texto_original, resumen, modelo, tokenizer):
    """Utiliza un modelo para puntuar automáticamente la calidad de un resumen."""

    prompt_juez = f"""Eres un juez estricto e imparcial evaluando resúmenes automáticos de hilos de foro.

TEXTO ORIGINAL:
{texto_original}

RESUMEN A EVALUAR:
{resumen}

CRITERIOS DE PUNTUACIÓN (Penaliza severamente los errores):
1. Precisión: Capta la idea central sin inventar información (cero alucinaciones).
2. Concisión: Va directo al grano, sin detalles irrelevantes..

ESCALA:
0-3: Inaceptable (Alucina, no tiene sentido o es una respuesta de chat).
4-6: Regular (Capta algo de la idea, pero es redundante o tiene muletillas).
7-8: Bueno (Preciso y conciso, con fallos mínimos).
9-10: Excelente (Abstractivo, directo y perfecto).

Regla estricta: No escribas absolutamente nada más. Da notas variadas según el criterio de evaluación.
Puntuación:"""


    # Tokenizar y generar respuesta limitando la longitud (solo queremos el número)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer(prompt_juez, return_tensors="pt", truncation=True, max_length=1500).to(device)
    outputs = modelo.generate(**inputs, max_new_tokens=5, temperature=0.1)

    respuesta = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extraer el número generado
    try:
        nota_str = respuesta.split("Puntuación:")[-1].strip()
        match = re.search(r'\b([0-9]|10)\b', nota_str)
        nota = int(match.group()) if match else 0
    except (ValueError, AttributeError):
        nota = None # Por si el modelo falla al dar solo un número

    return nota


In [ ]:
for i in range(10):
  resumen = lista_res_gemma[i][0]
  texto = textos_a_procesar[i]
  nota = evaluar_resumen(texto, resumen, model, tokenizer)
  print(f"Nota resumen {i+1}: {nota}")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Nota resumen 1: 0
Nota resumen 2: 0
Nota resumen 3: 0
Nota resumen 4: 9
Nota resumen 5: 10
Nota resumen 6: 9
Nota resumen 7: 10
Nota resumen 8: 0
Nota resumen 9: 0
Nota resumen 10: 0


In [ ]:
modelo_gemma = model

Vemos como el modelo ha dado la minima nota para la mayoria de los resumenes debido a que estos modelos no capturan bien la información y no la resumen bien. Pero aun asi debemos de tener en cuenta que este proceso de puntaje esta realizado por una ia que no conoce el contexto completo por lo que lo usamos solo de punto de vista externo, principalmente la evaluación del modelo se basa en lo explicado anteriormente donde vemos que la comprensión de ideas no es la correcta.

## mT5_multilingual_XLSum
Ahora pasamos a usar un modelos preentrenado como es mT5_multilingual_XLSum y evaluamos las diferencias encontradas.

Este modelos esta creado exclusivamente para resumir por lo que para esta tarea concreta va a ser un punto de inflexión.

También debemos tener en cuenta que debido al entrenamiento con texto procesado del dominio periodistico por lo que estará sobreajustado con esos parámetros

In [ ]:
import re
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

WHITESPACE_HANDLER = lambda k: re.sub('\s+', ' ', re.sub('\n+', ' ', k.strip()))


model_name = "csebuetnlp/mT5_multilingual_XLSum"
mt5_tok = AutoTokenizer.from_pretrained(model_name)
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_response_mt5(base_prompt,
                          texts,
                          model=mt5_model,
                          tokenizer=mt5_tok,
                          max_tokens=150,
                          num_beams=4):

    # 1. Configuración para Seq2Seq: El padding debe ir a la derecha.
    # mT5 ya tiene su propio pad_token configurado, no hace falta forzarlo.
    tokenizer.padding_side = "right"

    # Si se pasa un texto suelto, lo metemos en una lista
    if isinstance(texts, str):
        texts = [texts]

    # 2. Aplicar el prompt: mT5 no usa chat_templates (roles de usuario/asistente).
    # Simplemente concatenamos la instrucción con el texto.
    prompts = [ text.strip() for text in texts]

    # 3. Tokenizar el lote de textos
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024 # 1024 es más estable para la memoria de mT5
    ).to(model.device)

    # 4. Generar respuestas simultáneamente
    # Para resúmenes con mT5 funciona mucho mejor Beam Search que el muestreo (temperature/top_p)
    outputs = model.generate(
        **inputs,
        max_length=max_tokens,
        num_beams=num_beams,
        early_stopping=True
    )

    # 5. Decodificar todo el lote de golpe
    # A diferencia de los modelos CausalLM, mT5 SOLO devuelve el texto generado,
    # por lo que NO hay que recortar (hacer slicing) de los tokens de entrada.
    responses = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    return [resp.strip() for resp in responses]

<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_62179/3133677006.py:4: SyntaxWarning: invalid escape sequence '\s'
  WHITESPACE_HANDLER = lambda k: re.sub('\s+', ' ', re.sub('\n+', ' ', k.strip()))


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Volvemos a crear los diccionarios para evitar problemas de solapamiento tambien repetimos todos los pasos de la creación de resumenes y limpiado.

In [ ]:
import pandas as pd
subreddit_list = [result_elden_corregido, result_fortnite_corregido, result_walstreet_corregido, result_unpopular_corregido, result_nba_corregido, result_soccer_corregido]
# 1. Recopilar todas las submissions y preparar sus textos
todas_las_submissions = []
textos_a_procesar = []

for subreddit in subreddit_list:
    for submission in subreddit.get("submissions", []):
        title = submission.get("title_preprocessed") or ""
        description = submission.get("description") or ""

        comentarios_validos = [
            c.get("body_preprocessed") for c in submission.get("comments", [])
            if c.get("body_preprocessed") and c.get("body_preprocessed") != "[removed]" and c.get("author") != "AutoModerator"
        ]

        comments_text = ". ".join(comentarios_validos) + ". " if comentarios_validos else ""
        texto_completo = f"{title}. {description}. {comments_text}".strip()

        # Guardamos la referencia al diccionario original y el texto
        todas_las_submissions.append(submission)
        textos_a_procesar.append(texto_completo)

# 2. Procesar en lotes (batching) usando tu generate_response optimizado
batch_size = 8
lista_res_mt5 = []
#Hacemos bucle de unicamente los diez primeros hilos para poder compararlos, no añadimos mas por que tarda demasiado
for i in range(0, 10, batch_size):
    # Extraemos el bloque actual de textos y sus submissions
    lote_textos = textos_a_procesar[i : i + batch_size]
    lote_submissions = todas_las_submissions[i : i + batch_size]

    # Generamos los resúmenes de golpe en la GPU
    lote_resumenes = generate_response_mt5(zero_shot_prompt, lote_textos, max_tokens=100)

    # 3. Guardamos cada resumen directamente en su diccionario original
    for submission, resumen in zip(lote_submissions, lote_resumenes):

      resumen_limpiado, clasificacion = limpiar_resumen(resumen)
      submission['resumen_mt5'] = resumen_limpiado

      #guardamos los 10 primero resumenes de los hilos en una lista para evaluarlos a posteriori
      if len(lista_res_mt5) < 10:
        lista_res_mt5.append((resumen_limpiado, clasificacion))

In [ ]:
for i in range(10):
  print(f"Texto {i + 1}: {textos_a_procesar[i]}")

Texto 1: did your opinion on the dec change since launch would you rate it different now than in the first months. . at first i thought it was peak then i realized its even more peak then peak. it's good boss fights can be as fun as those of the base game however the lack of cut scenes make it seem like a bit of a rushed job nonetheless overall it was well done. i like the base game more. i played it back to back recently and the one thing i noticed on the and run is that the world did feel pretty empty relative to the base game everything else good or bad felt the same. same as launch many things added too few explained but it managed to make me emphasize with a character when the main game couldn't marinas story hits deep. no of of back then of of now. it is what a dec should be. still love it. over time it gets kinda tiring playing it the first time with a new character and having to collect the fragments but i like how fast you rush through it on no cycles expect for mira. i liked 

In [ ]:
for i in range(len(lista_res_mt5)):
  resumen = lista_res_mt5[i][0]
  clasificacion = lista_res_mt5[i][1]
  print(f"Resumen {i+1} ({clasificacion}): {resumen}")

Resumen 1 (aceptable): The dec has become a hugely popular game in the UK, but why has it changed?
Resumen 2 (aceptable): i do do build describes a worse feeling against soldiers of god rick and radars rune.
Resumen 3 (aceptable): This is a full transcript of the first night as Ellen Lord.
Resumen 4 (aceptable): i used a bleed build to beat malaria third try.
Resumen 5 (aceptable): When i started using the deaths poker, I asked anyone who wouldn't believe it.
Resumen 6 (aceptable): When i started playing a game in the early stages of the game, I was asked to explain why she refused to use the reliable weapon.
Resumen 7 (aceptable): i used mimic tear for almost every fight in the game but it wasn't enough to help me out.
Resumen 8 (aceptable): "yellow bastard from sin city" na a booklet of joker.
Resumen 9 (aceptable): German is a native language which sounds ultra cringe to me.
Resumen 10 (aceptable): When i started playing the Pokemon Go, I asked myself to explain why the game ended.


### Evaluacion
Tomando de nuevo como referencia el resumen 8, vemos que este modelos no llega a entender tampoco la idea del texto. Como mt5 usa muchas menos palabras y va mucho mas al grano no permite expresarse de manera airada sobre lo que cree que entiende del texto. Si es verdad que intenta coger la idea principal y plasmarla lo mas resumidamente posible frente a un contexto minimo, pero esta falta de expresividad o de coger información directamente del texto hace que los resumenes falten de información en la mayoria de los casos y muestren unicamente una idea de lo que se habla, no un resumen que te aporte ideas claves e información concreta relevante que aparece en el texto.

In [ ]:
for i in range(10):
  resumen = lista_res_mt5[i][0]
  texto = textos_a_procesar[i]
  nota = evaluar_resumen(texto, resumen, modelo_gemma, tokenizer)
  print(f"Nota resumen {i+1}: {nota}")

Nota resumen 1: 0
Nota resumen 2: 10
Nota resumen 3: 0
Nota resumen 4: 9
Nota resumen 5: 9
Nota resumen 6: 10
Nota resumen 7: 9
Nota resumen 8: 0
Nota resumen 9: 0
Nota resumen 10: 10


Aqui vemos finalmente que las notas que les ha dado el 'juez' han sido un poco mejores para este modelo, pero igualmente tiene mucho margen de mejora en algunos resumenes.

## Qwen2.5-1.5B-Instruct
De manera inicial queriamos usar LLama 3.2 pero devido a que su uso es restringido no podiamos usarlo hasta que nos aceptasen una solicitud y acabamos buscando modelos SLMs que pudiesen encajar, y aqui elegimos qwen.

Este modelo no necesita solicitar recursos y cumple estrictamente con el requisito de ser SLM. Es un modelo bien preparado para obedecer prompts restrictivos ZSL y tiene una tasa alta de cumplimiento.

In [ ]:
model_qwen_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer_qwen = AutoTokenizer.from_pretrained(model_qwen_id)

# Cargar modelo en la GPU
model_qwen = AutoModelForCausalLM.from_pretrained(
    model_qwen_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
import pandas as pd
subreddit_list = [result_elden_corregido, result_fortnite_corregido, result_walstreet_corregido, result_unpopular_corregido, result_nba_corregido, result_soccer_corregido]
# 1. Recopilar todas las submissions y preparar sus textos
todas_las_submissions = []
textos_a_procesar = []

for subreddit in subreddit_list:
    for submission in subreddit.get("submissions", []):
        title = submission.get("title_preprocessed") or ""
        description = submission.get("description") or ""

        comentarios_validos = [
            c.get("body_preprocessed") for c in submission.get("comments", [])
            if c.get("body_preprocessed") and c.get("body_preprocessed") != "[removed]" and c.get("author") != "AutoModerator"
        ]

        comments_text = ". ".join(comentarios_validos) + ". " if comentarios_validos else ""
        texto_completo = f"{title}. {description}. {comments_text}".strip()

        # Guardamos la referencia al diccionario original y el texto
        todas_las_submissions.append(submission)
        textos_a_procesar.append(texto_completo)

# 2. Procesar en lotes (batching) usando tu generate_response optimizado
batch_size = 8
lista_res_qwen = []
#Hacemos bucle de unicamente los diez primeros hilos para poder compararlos, no añadimos mas por que tarda demasiado
for i in range(0, 10, batch_size):
    # Extraemos el bloque actual de textos y sus submissions
    lote_textos = textos_a_procesar[i : i + batch_size]
    lote_submissions = todas_las_submissions[i : i + batch_size]

    # Generamos los resúmenes de golpe en la GPU
    lote_resumenes = generate_response(zero_shot_prompt, lote_textos,model= model_qwen, tokenizer=tokenizer_qwen,max_tokens=100)

    # 3. Guardamos cada resumen directamente en su diccionario original
    for submission, resumen in zip(lote_submissions, lote_resumenes):

      resumen_limpiado, clasificacion = limpiar_resumen(resumen)
      submission['resumen_qwen'] = resumen_limpiado

      #guardamos los 10 primero resumenes de los hilos en una lista para evaluarlos a posteriori
      if len(lista_res_qwen) < 10:
        lista_res_qwen.append((resumen_limpiado, clasificacion))

In [ ]:
for i in range(10):
  print(f"Texto {i + 1}: {textos_a_procesar[i]}")

Texto 1: did your opinion on the dec change since launch would you rate it different now than in the first months. . at first i thought it was peak then i realized its even more peak then peak. it's good boss fights can be as fun as those of the base game however the lack of cut scenes make it seem like a bit of a rushed job nonetheless overall it was well done. i like the base game more. i played it back to back recently and the one thing i noticed on the and run is that the world did feel pretty empty relative to the base game everything else good or bad felt the same. same as launch many things added too few explained but it managed to make me emphasize with a character when the main game couldn't marinas story hits deep. no of of back then of of now. it is what a dec should be. still love it. over time it gets kinda tiring playing it the first time with a new character and having to collect the fragments but i like how fast you rush through it on no cycles expect for mira. i liked 

In [ ]:
for i in range(len(lista_res_qwen)):
  resumen = lista_res_qwen[i][0]
  clasificacion = lista_res_qwen[i][1]
  print(f"Resumen {i+1} ({clasificacion}): {resumen}")

Resumen 1 (no aceptable): The reviewer discusses their opinions on the Destiny Dec, noting initial confusion about its difficulty compared to Base Game, enjoyment of certain features (like boss battles), and appreciation for exploration and storytelling elements. They express mixed feelings regarding the expanded content, mentioning improved crafting system but criticism of repetitive missions, fragmented lore, and perceived over-reliance on grinding. The review highlights significant changes such as reduced cutscenes, fewer story events, and increased focus on gameplay mechanics beyond narrative depth. Despite acknowledging improvements in combat challenges
Resumen 2 (aceptable): The passage discusses various difficulties encountered during gaming experiences, including issues with emote-free emotes packs, inconsistent performance of character units (like Malaria), frustration with bosses who remain alive beyond their expected health, and overall difficulty levels across different gam

### Evaluacion
Volvemos a repetir el analisis del resumen 8. Esta vez existe una mejor comprensión de la idea del contexto que es la aparición de diversos nombres, sin embargo, no llega a entender por que y asi lo refleja en el resumen donde habla sobre la falta de contexto que aporta. Este modelo representa según vemos una mejora en la generabilidad de gemma y puede llegar a ser mejor comprensión de texto que mt5 pero para eso deberiamos de usar muchas mas pruebas y mas textos mas completos y ricos para saberlo.

In [ ]:
for i in range(10):
  resumen = lista_res_qwen[i][0]
  texto = textos_a_procesar[i]
  nota = evaluar_resumen(texto, resumen, modelo_gemma, tokenizer)
  print(f"Nota resumen {i+1}: {nota}")

Nota resumen 1: 0
Nota resumen 2: 0
Nota resumen 3: 1
Nota resumen 4: 9
Nota resumen 5: 9
Nota resumen 6: 9
Nota resumen 7: 10
Nota resumen 8: 0
Nota resumen 9: 0
Nota resumen 10: 8


Las notas conseguidas son practicamente las mismas que las que tenemos en gemma por lo que podriamos decir que son modelos muy parecidos.

## Conclusion
Como analisis final creemos que gemma puede llegar a hacer resumenes pero tiene una idea de resumen mas extractiva que generativa ya que coge demasiada información bruta del texto en vez de buscar enfoques para la explicación.

Podemos observar como mt5 tiene un enfoque mucho mas generativo que gemma. Este segundo modelo hace resumenes mas imaginativos y menos regidos a las ideas concretas del texto lo que les hace ser muchos mas cortos y concisos.

Qwen vemos que es mucho mas abstracto y generativo que gemma lo que le hace sacar ideas y textos mas imaginativos. Sin embargo, no tiene la capacidad de resumir que tiene mt5.

Uno de los problemas que vemos es que la maquina no conoce el contexto de lo que le pasamos. No conoce que estos textos son submissions de reddit donde el principio comprende el titulo y descripción del submission, y que todo lo demás son respuestas de usuarios. Además son textos que aunque para nosotros puedan ser tediosos de leer para una maquina o un modelo pueden ser insuficientes a la hora de sacar información y realizar un contexto claro que les ayude a analizar la situación.

Finalmente, la diferencia clave entre los modelos preentrenados y los SLM es que mt5 va al grano sin necesidad de explicaciones como prompts ,mientras que los modelos SLMs necesitan que les aportes un buen prompt para poder hacer su tarea algo que puede llevar a tener resumenes completamente diferentes unicamente por un fallo de expresión en el prompt. También debemos tener en cuenta que mt5 es un modelo preentrenado con texto concreto para prepararlo por lo que tiene una preparación que los otros modelos no tienen, pero esta preparación le puede llevar a alucinaciones sobre el texto del entrenamiento que no queremos tener.


In [ ]:
with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_wallstreet_resumenes", "w", encoding="utf-8") as f:
    json.dump(result_walstreet_corregido, f, ensure_ascii=False, indent=2)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_elden_resumenes", "w", encoding="utf-8") as f:
    json.dump(result_elden_corregido, f, ensure_ascii=False, indent=2)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_fortnite_resumenes", "w", encoding="utf-8") as f:
    json.dump(result_fortnite_corregido, f, ensure_ascii=False, indent=2)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_unpopular_resumenes", "w", encoding="utf-8") as f:
    json.dump(result_unpopular_corregido, f, ensure_ascii=False, indent=2)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_nba_resumenes", "w", encoding="utf-8") as f:
    json.dump(result_nba_corregido, f, ensure_ascii=False, indent=2)

with open("/content/drive/MyDrive/lenguaje_natural/entregable/datos/result_soccer_resumenes", "w", encoding="utf-8") as f:
    json.dump(result_soccer_corregido, f, ensure_ascii=False, indent=2)

#Como unicamente hemos hecho resumenes de los diez primeros hilos debido a que tarda demasiado, los resumenes unicamente se verán disponibles en el archivo de result_walstreet_resumenes en los diez primeros hilos